# Skill-Scoring Dataset — Short EDA

Quick look at the synthetic corpus that feeds the scoring model:

1. **Data variance** — label balance, structural spread, document lengths
2. **Flags summary** — allocation rationality, skill-showcase deltas, shortcut checks
3. **Persona-level skill flagging** — drop a `(persona, skill)` row when, across that
   persona's documents, `max |delta| >= 3` **or** `avg |delta| >= 2`
4. **Other findings**

Training unit = one `(persona, skill)` row (label = the persona's global skill level).
Run top-to-bottom with the project `.venv` kernel from the repo root.

In [1]:
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd
pd.set_option('display.max_rows', 60)

# display() is provided by the IPython kernel; define a fallback so the name
# resolves (and the notebook still runs) outside a notebook context too.
try:
    from IPython.display import display
except ImportError:
    def display(*objs):
        for o in objs:
            print(o)

DATA = Path('data')
personas = json.load(open(DATA / 'personas.json', encoding='utf-8'))
db = json.load(open(DATA / 'documents_db.json', encoding='utf-8'))

def load_report(name):
    p = DATA / 'reports' / name
    return json.load(open(p, encoding='utf-8')) if p.exists() else None

showcase  = load_report('skill_showcase_report.json')
alloc_rep = load_report('allocation_rationality_report.json')
shortcut  = load_report('shortcut_check_report.json')

def norm(s):
    return s.strip().lower().replace(' ', '_').replace('/', '_').replace('-', '_')

doc_persona = {d['doc_id']: d['persona_id'] for d in db.values()}
docs_by_persona = defaultdict(list)
for d in db.values():
    docs_by_persona[d['persona_id']].append(d)

def dist(values):
    s = pd.Series(values).value_counts(dropna=False)
    out = s.rename('count').to_frame()
    out['pct'] = (100 * out['count'] / out['count'].sum()).round(1)
    return out

print(f"{len(personas)} personas | {len(db)} documents")

310 personas | 1192 documents


## 1. Data variance

In [2]:
# Training rows = one (persona, skill) pair
rows = [(p['persona_id'], sk, lv) for p in personas for sk, lv in p['skills'].items()]
rows_df = pd.DataFrame(rows, columns=['persona_id', 'skill', 'label'])
rows_df['nskill'] = rows_df['skill'].map(norm)
print(f"Training rows (persona x skill): {len(rows_df)}")

# Label / proficiency distribution -- the class balance the scorer must learn
label_dist = rows_df['label'].value_counts().sort_index().rename('count').to_frame()
label_dist['pct'] = (100 * label_dist['count'] / len(rows_df)).round(1)
label_dist

Training rows (persona x skill): 3478


,count,pct
label,,
1,2,0.1
2,198,5.7
3,1521,43.7
4,1260,36.2
5,497,14.3


In [3]:
# Structural spread: skills per persona, documents per persona
skills_per = pd.Series([len(p['skills']) for p in personas], name='skills_per_persona')
docs_per = pd.Series([len(docs_by_persona[p['persona_id']]) for p in personas], name='docs_per_persona')
pd.concat([skills_per.describe(), docs_per.describe()], axis=1).round(2)

,skills_per_persona,docs_per_persona
count,310.00,310.00
mean,11.22,3.85
std,2.12,1.36
min,8.00,0.00
25%,10.00,3.00
50%,11.00,4.00
75%,12.00,5.00
max,20.00,7.00


In [4]:
# Categorical variance: doc types, archetypes, seniority
print('--- document types ---');  display(dist([d['type'] for d in db.values()]))
print('--- archetypes ---');      display(dist([p['hyperparams'].get('archetype') for p in personas]))
print('--- seniority ---');       display(dist([p['hyperparams'].get('seniority') for p in personas]))

--- document types ---


,count,pct
project_readme,440,36.9
cv,309,25.9
recommendation,237,19.9
blog,137,11.5
linkedin,69,5.8


--- archetypes ---


,count,pct
frontend_engineer,35,11.3
backend_engineer,29,9.4
fullstack_developer,28,9.0
devops_sre,26,8.4
ml_engineer,25,8.1
platform_engineer,23,7.4
security_engineer,22,7.1
qa_test_engineer,20,6.5
data_engineer,19,6.1
data_scientist,18,5.8


--- seniority ---


,count,pct
mid,88,28.4
junior,82,26.5
senior,82,26.5
staff,58,18.7


In [5]:
# Document length variance (words) by type
len_df = pd.DataFrame([{'type': d['type'], 'words': len(d.get('text', '').split())}
                       for d in db.values()])
len_df.groupby('type')['words'].describe()[['count', 'mean', 'std', 'min', 'max']].round(0)

,count,mean,std,min,max
type,,,,,
blog,137.0,372.0,42.0,284.0,474.0
cv,309.0,501.0,94.0,250.0,753.0
linkedin,69.0,176.0,31.0,113.0,305.0
project_readme,440.0,298.0,48.0,43.0,449.0
recommendation,237.0,221.0,29.0,128.0,322.0


In [6]:
# Does proficiency vary by role? (mean label per archetype)
arch_of = {p['persona_id']: p['hyperparams'].get('archetype') for p in personas}
rows_df['archetype'] = rows_df['persona_id'].map(arch_of)
rows_df.groupby('archetype')['label'].agg(['mean', 'std', 'count']).round(2).sort_values('mean')

,mean,std,count
archetype,,,
qa_test_engineer,3.20,0.85,200
product_manager,3.28,0.70,151
data_engineer,3.46,0.75,226
ml_engineer,3.51,0.84,283
mobile_developer,3.52,0.70,158
fullstack_developer,3.55,0.81,354
platform_engineer,3.56,0.80,266
embedded_engineer,3.58,0.85,127
game_developer,3.59,0.79,69


## 2. Flags summary

In [7]:
# Allocation rationality -- whole-persona evidence coherence (LLM judged)
flagged_personas = [r['persona_id'] for r in alloc_rep['results'] if r.get('flagged')]
print(f"Allocation rationality: {len(flagged_personas)}/{alloc_rep['total_personas']} personas flagged "
      f"({100 * len(flagged_personas) / alloc_rep['total_personas']:.1f}%)")
print('rationality score distribution (1-5):')
dist([r['score'] for r in alloc_rep['results'] if 'score' in r]).sort_index()

Allocation rationality: 14/309 personas flagged (4.5%)
rationality score distribution (1-5):


,count,pct
0,1,0.3
1,2,0.6
2,12,3.9
3,68,22.0
4,115,37.2
5,111,35.9


In [8]:
# Skill-showcase comparisons (per doc x skill): allocated vs LLM-judged level
cmp = []
for r in showcase['results']:
    pid = doc_persona.get(r['doc_id'])
    for c in r['comparisons']:
        cmp.append({'persona_id': pid, 'doc_id': r['doc_id'], 'skill': norm(c['skill']),
                    'allocated': c['allocated_level'], 'judged': c['llm_judged_level'],
                    'abs_delta': abs(c['delta'])})
cmp_df = pd.DataFrame(cmp)
print(f"Total doc x skill comparisons: {len(cmp_df)}")

ad = cmp_df['abs_delta'].value_counts().sort_index().rename('count').to_frame()
ad['pct'] = (100 * ad['count'] / len(cmp_df)).round(1)
print('abs-delta distribution:'); display(ad)

# Flag rate at candidate thresholds
thr = pd.DataFrame([{'rule': f'|delta|>={t}',
                     'flagged': int((cmp_df['abs_delta'] >= t).sum()),
                     'pct': round(100 * (cmp_df['abs_delta'] >= t).mean(), 1)}
                    for t in (2, 3, 4)])
print('flag rate at thresholds (comparison grain):'); display(thr)

Total doc x skill comparisons: 10690
abs-delta distribution:


,count,pct
abs_delta,,
0,4356,40.7
1,4824,45.1
2,1182,11.1
3,258,2.4
4,70,0.7


flag rate at thresholds (comparison grain):


,rule,flagged,pct
0,|delta|>=2,1510,14.1
1,|delta|>=3,328,3.1
2,|delta|>=4,70,0.7


In [9]:
# Where do the disputes concentrate? Flag rate by allocated level
g = cmp_df.assign(f2=cmp_df['abs_delta'] >= 2, f3=cmp_df['abs_delta'] >= 3)
by_lv = g.groupby('allocated').agg(n=('abs_delta', 'size'), flag_ge2=('f2', 'sum'), flag_ge3=('f3', 'sum'))
by_lv['pct_ge2'] = (100 * by_lv['flag_ge2'] / by_lv['n']).round(1)
by_lv['pct_ge3'] = (100 * by_lv['flag_ge3'] / by_lv['n']).round(1)
by_lv

,n,flag_ge2,flag_ge3,pct_ge2,pct_ge3
allocated,,,,,
2,1478,89,6,6.0,0.4
3,4962,702,0,14.1,0.0
4,3154,331,227,10.5,7.2
5,1096,388,95,35.4,8.7


In [10]:
# Non-LLM shortcut checks
bp = shortcut['banned_phrases']
print(f"Banned phrases: {bp['documents_with_issues']}/{bp['total_documents']} docs flagged "
      f"({bp['total_issues']} issues)")
print(f"Level-1 absence violations: {shortcut['level1_absence']['violations']}")
bb = shortcut['bow_baseline']
print(f"Bag-of-words baseline CV accuracy: {bb['mean_cv_accuracy']:.3f} "
      f"(threshold {bb['threshold']}, passed={bb['passed']})")
print("  low BoW accuracy => labels are NOT trivially separable by keywords (good)")

Banned phrases: 5/1192 docs flagged (6 issues)
Level-1 absence violations: 0
Bag-of-words baseline CV accuracy: 0.395 (threshold 0.45, passed=True)
  low BoW accuracy => labels are NOT trivially separable by keywords (good)


## 3. Persona-level skill flagging

Aggregate the per-document `abs_delta` for each skill **up to the persona**, then drop the
`(persona, skill)` training row when `max |delta| >= 3` **or** `avg |delta| >= 2`.
The `max` clause catches a single egregious mis-showcase; the `avg` clause catches a skill
that is mildly-but-consistently off across documents.

In [11]:
# max / avg abs-delta per (persona, skill)
agg = (cmp_df.groupby(['persona_id', 'skill'])['abs_delta']
       .agg(max_abs='max', avg_abs='mean', n_docs='size').reset_index())
agg['flagged'] = (agg['max_abs'] >= 3) | (agg['avg_abs'] >= 2)

# Join onto the real training rows (never-judged skills are kept)
merged = rows_df.merge(agg, left_on=['persona_id', 'nskill'], right_on=['persona_id', 'skill'],
                       how='left', suffixes=('', '_agg'))
merged['flagged'] = merged['flagged'].fillna(False)

n_drop = int(merged['flagged'].sum())
print(f"Rule: drop (persona, skill) if max|delta|>=3 OR avg|delta|>=2")
print(f"Dropped {n_drop}/{len(merged)} rows = {100 * n_drop / len(merged):.1f}%  |  "
      f"kept {len(merged) - n_drop}")

Rule: drop (persona, skill) if max|delta|>=3 OR avg|delta|>=2
Dropped 310/3478 rows = 8.9%  |  kept 3168


In [12]:
# Drop impact per class (label)
tbl = merged.groupby('label').agg(before=('flagged', 'size'), dropped=('flagged', 'sum'))
tbl['after'] = tbl['before'] - tbl['dropped']
tbl['dropped_pct'] = (100 * tbl['dropped'] / tbl['before']).round(1)
tbl[['before', 'after', 'dropped', 'dropped_pct']]

,before,after,dropped,dropped_pct
label,,,,
1,2,2,0,0.0
2,198,187,11,5.6
3,1521,1499,22,1.4
4,1260,1111,149,11.8
5,497,369,128,25.8


In [13]:
# Worst offenders (highest max/avg abs-delta)
agg.sort_values(['max_abs', 'avg_abs'], ascending=False).head(10).reset_index(drop=True)

,persona_id,skill,max_abs,avg_abs,n_docs,flagged
0,p_064,message_queues,4,4.00,1,True
1,p_140,python,4,4.00,1,True
2,p_226,terraform,4,4.00,1,True
3,p_248,ios_development,4,4.00,1,True
4,p_123,javascript,4,3.50,4,True
5,p_306,project_management,4,3.00,4,True
6,p_306,stakeholder_communication,4,3.00,4,True
7,p_117,javascript,4,2.80,5,True
8,p_039,tensorflow,4,2.75,4,True
9,p_046,linux_administration,4,2.75,4,True


## 4. Other findings

In [14]:
# Data-quality note: skill_evidence double-lists each skill (case variants 'aws' vs 'AWS')
dup_docs = sum(1 for d in db.values()
               if len({k.lower() for k in d.get('skill_evidence', {})}) != len(d.get('skill_evidence', {})))
print(f"Documents whose skill_evidence has case-duplicate keys: {dup_docs}/{len(db)}")
print("skill_evidence uses normalized keys (e.g. 'ci_cd_pipelines') while persona.skills")
print("uses display keys (e.g. 'CI/CD pipelines') -- norm() bridges them. Only a couple of")
print("docs additionally store both case variants, a minor source-side inconsistency.")

Documents whose skill_evidence has case-duplicate keys: 2/1192
skill_evidence uses normalized keys (e.g. 'ci_cd_pipelines') while persona.skills
uses display keys (e.g. 'CI/CD pipelines') -- norm() bridges them. Only a couple of
docs additionally store both case variants, a minor source-side inconsistency.


In [15]:
# Shortcut risk: is document length a giveaway of skill level?
from scipy.stats import spearmanr
pairs = [(inten, len(d.get('text', '').split()))
         for d in db.values() for inten in d.get('skill_evidence', {}).values()]
pe = pd.DataFrame(pairs, columns=['intensity', 'words'])
rho, p = spearmanr(pe['intensity'], pe['words'])
print(f"Spearman(skill intensity, doc word-count) = {rho:.3f}  (p={p:.1e}, n={len(pe)})")
print("low-to-moderate => length is not a strong shortcut for proficiency.")

Spearman(skill intensity, doc word-count) = 0.266  (p=1.8e-217, n=13485)
low-to-moderate => length is not a strong shortcut for proficiency.


### Takeaways
- **Class imbalance is the headline variance issue**: labels 3–4 dominate; level 5 (and the
  near-absent level 1 at persona-skill grain) are thin — weight the loss or rebalance.
- **Flags cluster at high allocated levels**: the judge disputes level-5 claims far more often,
  so any delta-based filter disproportionately thins the rarest class.
- **The `max>=3 or avg>=2` rule drops ~9% of rows**, concentrated on labels 4–5 — a middle ground
  between the loose `|delta|>=2` (~31%) and strict `|delta|>=3` (~7%) cuts.
- Shortcut checks pass (low BoW accuracy, weak length↔level correlation), so labels aren't
  trivially guessable — the model has to actually read the evidence.